# M11: Pipeline Automation — End-to-End SageMaker Pipeline

**Pipeline Position:** Full Pipeline: End-to-End Automation
**Input S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m1/` *(real — consumes M1's selected_scenes.json, read-only)*
**Output S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m11/`
**Notebook instance:** `ml.t3.medium` (orchestration only — authors & submits the pipeline)
**Step instances:** `ml.m5.xlarge` × 3 (CPU) — the actual processing steps

## What This Module Does

Defines and **executes** a real SageMaker Pipeline that chains three AV data
processing stages as a dependency DAG, then polls it to completion and records
the run — this is genuine `upsert()` + `start()`, not a definition-only demo:

1. **Step 1: Captioning** (ProcessingStep) — caption each M1-selected nuScenes scene
2. **Step 2: Curation** (ProcessingStep) — filter + deduplicate captions
3. **Step 3: Augmentation** (ProcessingStep) — augment curated captions

## Pipeline DAG

```
┌────────────────────┐     ┌────────────────────┐     ┌────────────────────┐
│  Step 1: Caption   │───► │  Step 2: Curate    │───► │  Step 3: Augment   │
│   (ml.m5.xlarge)   │     │   (ml.m5.xlarge)   │     │   (ml.m5.xlarge)   │
│                    │     │                    │     │                    │
│ In:  m1/ (real)    │     │ In:  step1 output  │     │ In:  step2 output  │
│ Out: m11/pipeline/ │     │ Out: m11/pipeline/ │     │ Out: m11/pipeline/ │
│      step1_caption │     │      step2_curate  │     │      step3_augment │
└────────────────────┘     └────────────────────┘     └────────────────────┘
```

*Step 1 reads M1's real output. All step outputs stay inside M11's own
`m11/pipeline/` namespace so the demo never overwrites the real M2/M3/M4 module
outputs (those use a richer schema that M3/M8 depend on).*

## CPU by design — the point is orchestration, not compute

The three step scripts are **pure Python over M1's scene metadata** (JSON), so
they need no GPU. We run them on CPU (`ml.m5.xlarge`, which has processing-job
quota here — the g5* processing quota is 0) so the pipeline actually runs, cheaply.
**What M11 teaches is the orchestration pattern** — a reproducible dependency DAG,
per-step instances that start/stop automatically, and full lineage — which is
identical whether a step runs on CPU or GPU. In production, the captioning step
would swap to a GPU image running a real VLM (e.g. Cosmos Reason); only the
per-step compute changes, not the pipeline. See
[`docs/PIPELINE_M11.md`](../docs/PIPELINE_M11.md).

## Benefits of Pipeline Automation

- **Reproducibility** — Same steps, same order, every time
- **Fault tolerance** — Automatic retries on transient failures
- **Cost control** — Instances start/stop per step (no idle time)
- **Lineage tracking** — Full audit trail of inputs, outputs, parameters
- **Scheduling** — Trigger on new data arrival or cron schedule

### One-time environment setup — expect this, it is not an error

The first code cell pins the classic **SageMaker SDK v2** (this notebook uses the v2 API; the kernel may ship v3). On a fresh kernel it installs v2 and continues with **no restart**. Only if v3 was already loaded into memory will the kernel restart **once** — you'll see a small orange banner (not a red error). When it returns, just choose **Run All** again; the cell detects v2 and continues.

> **Note on the M1->M11 link:** Step 1 captions the scenes M1 selected, so the link is real *when M1 has run*. If `users/<profile>/m1/` is missing the expected JSON, the step logs a warning and falls back to a few synthetic scenes — run M1 first for a genuine end-to-end pass.

In [ ]:
"""Environment Setup — SDK pin (run this cell first, on its own)"""
# The SageMaker Distribution kernel may ship the NEW SageMaker Python SDK **v3**
# (modular: sagemaker.core / sagemaker.train, no top-level Session or the classic
# sagemaker.pytorch / sagemaker.workflow layout). This notebook uses the classic
# **v2** API, so we pin v2 (security-patched >=2.257.2).
#
# We restart the kernel ONLY if v3 was already imported into memory (pip can swap
# files on disk, but a resident module stays in memory). On a fresh kernel where
# sagemaker was never imported, we install v2 and import it in-place — NO restart.
# If a restart IS needed you'll see a small orange banner (not a red error); when
# the kernel comes back, just Run All again — this cell detects v2 and continues.
import subprocess, sys


def _sm_version():
    """Return (version, resident) WITHOUT importing sagemaker.

    Importing to detect would pull v3 into memory as a side effect and force an
    unnecessary restart, so we probe sys.modules (residency) + importlib.metadata
    (on-disk version) instead.
    """
    mod = sys.modules.get("sagemaker")
    if mod is not None:
        return getattr(mod, "__version__", None), True          # resident in memory
    try:
        import importlib.metadata as _md
        return _md.version("sagemaker"), False                  # on disk, not imported
    except Exception:
        return None, False


_ver, _resident = _sm_version()
_need_v2 = (_ver is None) or (not str(_ver).startswith("2"))
_restarting = False

if _need_v2:
    print(f"Current SageMaker SDK: {_ver} -> installing v2 (>=2.257.2,<3) ...")
    _res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "sagemaker>=2.257.2,<3"],
        capture_output=True, text=True,
    )
    if _res.returncode != 0:
        # Real failure: surface everything (do NOT swallow) and stop loudly.
        print(_res.stdout)
        print(_res.stderr)
        raise RuntimeError("pip install of sagemaker v2 failed - see output above.")
    # Success: pip's harmless dependency-resolver conflict wall stayed captured.

    if _resident and str(_ver).startswith("3"):
        # v3 objects are already in memory - a one-time restart is the only clean
        # way to evict them. Friendly banner, graceful restart, NO raised exception.
        from IPython.display import display, HTML
        display(HTML(
            '<div style="padding:10px;border-left:4px solid #FF9900;'
            'background:#FFF8E1;font-family:sans-serif">'
            '<b>One-time kernel restart</b> to load the pinned SageMaker SDK v2. '
            'When it returns (a few seconds), just <b>Run All</b> again - this cell '
            'will detect v2 and continue. This is expected, not an error.</div>'))
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
        _restarting = True   # skip the rest of this cell; the queued restart fires
    # else: nothing resident -> import the freshly installed v2 in-place, no restart.

if not _restarting:
    import os
    import json
    import time
    from pathlib import Path
    from datetime import datetime, timezone

    import boto3
    import sagemaker
    from sagemaker.session import Session                                  # v2 layout
    from sagemaker.workflow.pipeline import Pipeline
    from sagemaker.workflow.steps import ProcessingStep
    from sagemaker.workflow.pipeline_context import PipelineSession
    from sagemaker.workflow.parameters import ParameterString
    from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

    print(f"SageMaker SDK v2 confirmed: {sagemaker.__version__}")

    # --- S3 Path Configuration ---
    ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
    PROFILE = os.environ.get("USER_PROFILE", "default")

    USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")
    SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
    OUTPUT_PREFIX = f"users/{PROFILE}/m11/"

    # SageMaker sessions. The execution role can only write under the user-workspace
    # bucket's users/* prefix. By default the SDK uploads step code + the pipeline
    # definition to sagemaker-<region>-<account> at the bucket root — both outside the
    # role's scope → AccessDenied. So pin BOTH the default bucket AND a
    # default_bucket_prefix of users/<profile>/m11 so every SDK upload lands where the
    # role can write.
    _BUCKET_PREFIX = f"users/{PROFILE}/m11"
    sagemaker_session = Session(default_bucket=USER_BUCKET, default_bucket_prefix=_BUCKET_PREFIX)
    pipeline_session = PipelineSession(default_bucket=USER_BUCKET, default_bucket_prefix=_BUCKET_PREFIX)
    role = sagemaker.get_execution_role()
    region = sagemaker_session.boto_region_name

    # Pipeline configuration
    PIPELINE_NAME = f"av30-data-pipeline-{PROFILE}"

    # --- Clients ---
    s3 = boto3.client("s3")
    sm_client = boto3.client("sagemaker")

    print(f"Account ID: {ACCOUNT_ID}")
    print(f"Profile: {PROFILE}")
    print(f"Region: {region}")
    print(f"Role: {role}")
    print(f"Pipeline name: {PIPELINE_NAME}")
    print(f"Upload bucket/prefix: {USER_BUCKET}/{_BUCKET_PREFIX}")
    print(f"SageMaker SDK: {sagemaker.__version__}")

In [ ]:
"""Create processing scripts for each pipeline step"""

SCRIPTS_DIR = Path("/tmp/m11_pipeline_scripts")
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Step 1: Captioning Script ---
# Consumes M1's REAL output: users/<profile>/m1/selected_scenes.json (the nuScenes
# scenes M1 selected, each with a `name` + human `description`) and manifest.json
# (frame counts). M1 does NOT copy image files into m1/ — it records the scene
# metadata and points at the shared dataset — so we caption from that metadata,
# which makes the M1->M11 data link real (not a synthetic count from thin air).
caption_script = '''
"""Step 1: AV Video Captioning (pipeline demo)
Generates one caption per M1-selected nuScenes scene, grounded in that scene's
real name + description. In production this is where a VLM (e.g. Cosmos Reason)
would caption the actual CAM_FRONT frames; for the orchestration demo we derive
deterministic captions from M1's scene metadata so the pipeline runs CPU-only.
"""
import os
import json
from pathlib import Path
from datetime import datetime, timezone

INPUT_DIR = Path("/opt/ml/processing/input")
OUTPUT_DIR = Path("/opt/ml/processing/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Step 1: Captioning")
print(f"Input dir: {INPUT_DIR}")

# Load M1's selected scenes (real metadata). Fall back gracefully if absent.
scenes = []
scenes_file = INPUT_DIR / "selected_scenes.json"
manifest_file = INPUT_DIR / "manifest.json"
if scenes_file.exists():
    scenes = json.loads(scenes_file.read_text())
    print(f"Loaded {len(scenes)} M1-selected scenes from selected_scenes.json")
elif manifest_file.exists():
    mani = json.loads(manifest_file.read_text())
    scenes = [{"name": n, "description": ""} for n in mani.get("scenes", [])]
    print(f"selected_scenes.json missing; used {len(scenes)} scene names from manifest.json")
else:
    # last-resort synthetic fallback so the step still produces output
    scenes = [{"name": f"synthetic-{i}", "description": ""} for i in range(3)]
    print(f"No M1 output found; using {len(scenes)} synthetic scenes")

captions = []
for i, sc in enumerate(scenes):
    name = sc.get("name", f"scene-{i}")
    desc = (sc.get("description") or "").strip()
    # Ground the caption in the real scene description when present.
    if desc:
        caption = (f"Scene {name}: {desc}. Urban driving scene with vehicles, "
                   f"lane markings, and road users as annotated in nuScenes.")
    else:
        caption = (f"Scene {name}: urban road with vehicles and pedestrians, "
                   f"lane markings visible, traffic flowing normally.")
    captions.append({
        "frame_idx": i,
        "scene": name,
        "caption": caption,
        "source_description": desc,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    })

output_data = {
    "module": "M2_Captioning_PipelineStep",
    "note": "Captions derived from M1 scene metadata (demo). Production: VLM over CAM_FRONT frames.",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "num_captions": len(captions),
    "captions": captions,
}

output_path = OUTPUT_DIR / "captions.json"
output_path.write_text(json.dumps(output_data, indent=2))
print(f"Written {len(captions)} captions to {output_path}")
'''

# --- Step 2: Curation Script ---
curation_script = '''
"""Step 2: Data Curation
Filters low-quality captions and removes duplicates.
"""
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

INPUT_DIR = Path("/opt/ml/processing/input")
OUTPUT_DIR = Path("/opt/ml/processing/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Step 2: Curation")

# Load captions from previous step
captions_file = INPUT_DIR / "captions.json"
with open(captions_file) as f:
    input_data = json.load(f)

captions = input_data["captions"]
print(f"Loaded {len(captions)} captions")

# Quality filtering
QUALITY_THRESHOLD = 0.4

def score_caption(caption_text):
    word_count = len(caption_text.split())
    av_keywords = ["vehicle", "road", "lane", "traffic", "pedestrian"]
    keyword_score = sum(1 for kw in av_keywords if kw in caption_text.lower()) / 5.0
    length_score = min(word_count / 50, 1.0)
    return 0.5 * keyword_score + 0.5 * length_score

filtered = [c for c in captions if score_caption(c["caption"]) >= QUALITY_THRESHOLD]
print(f"After quality filter: {len(filtered)} captions")

# Deduplication
seen = set()
deduped = []
for c in filtered:
    h = hashlib.md5(c["caption"].encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        deduped.append(c)

print(f"After deduplication: {len(deduped)} captions")

output_data = {
    "module": "M3_Curation_PipelineStep",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "curation_stats": {
        "input_count": len(captions),
        "quality_filtered": len(captions) - len(filtered),
        "duplicates_removed": len(filtered) - len(deduped),
        "output_count": len(deduped)
    },
    "curated_captions": deduped
}

output_path = OUTPUT_DIR / "curated_captions.json"
output_path.write_text(json.dumps(output_data, indent=2))
print(f"Written {len(deduped)} curated captions to {output_path}")
'''

# --- Step 3: Augmentation Script ---
augmentation_script = '''
"""Step 3: Data Augmentation
Augments curated captions with paraphrases and variations.
"""
import os
import json
import random
from pathlib import Path
from datetime import datetime, timezone

INPUT_DIR = Path("/opt/ml/processing/input")
OUTPUT_DIR = Path("/opt/ml/processing/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Step 3: Augmentation")

# Load curated captions
curated_file = INPUT_DIR / "curated_captions.json"
with open(curated_file) as f:
    input_data = json.load(f)

curated = input_data["curated_captions"]
print(f"Loaded {len(curated)} curated captions")

# Augmentation strategies
AUGMENTATION_TEMPLATES = [
    "In this driving scene: {caption}",
    "The autonomous vehicle observes: {caption}",
    "Scene analysis: {caption}",
    "Perception output: {caption}"
]

random.seed(42)  # deterministic augmentation -> reproducible outputs
augmented = []
for cap in curated:
    # Original
    augmented.append({**cap, "augmentation": "original"})
    
    # Generate 2 augmented versions per original
    templates = random.sample(AUGMENTATION_TEMPLATES, 2)
    for template in templates:
        aug_cap = {**cap}
        aug_cap["caption"] = template.format(caption=cap["caption"])
        aug_cap["augmentation"] = template.split("{")[0].strip()
        augmented.append(aug_cap)

factor = (len(augmented) / len(curated)) if curated else 0.0
print(f"Augmented: {len(curated)} -> {len(augmented)} captions ({factor:.1f}x)")

output_data = {
    "module": "M4_Augmentation_PipelineStep",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "augmentation_stats": {
        "input_count": len(curated),
        "output_count": len(augmented),
        "augmentation_factor": round(factor, 1),
        "strategies_used": len(AUGMENTATION_TEMPLATES)
    },
    "augmented_captions": augmented
}

output_path = OUTPUT_DIR / "augmented_captions.json"
output_path.write_text(json.dumps(output_data, indent=2))
print(f"Written {len(augmented)} augmented captions to {output_path}")
print("Pipeline complete!")
'''

# Write scripts to disk
(SCRIPTS_DIR / "step1_caption.py").write_text(caption_script)
(SCRIPTS_DIR / "step2_curate.py").write_text(curation_script)
(SCRIPTS_DIR / "step3_augment.py").write_text(augmentation_script)

print("Pipeline scripts created:")
for f in sorted(SCRIPTS_DIR.glob("*.py")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")

In [ ]:
"""Define SageMaker Pipeline with 3 processing steps"""

# Pipeline parameters (configurable at execution time)
param_profile = ParameterString(name="UserProfile", default_value=PROFILE)
param_input_bucket = ParameterString(name="InputBucket", default_value=USER_BUCKET)

# --- S3 paths -------------------------------------------------------------
# Step 1 READS M1's real output (users/<profile>/m1/) — read-only, never modified.
# The step OUTPUTS go into an M11-PRIVATE namespace (users/<profile>/m11/pipeline/)
# on purpose: the step scripts write files named captions.json / curated_captions.json
# whose keys are M11 demo stubs (frame_idx/scene/caption — NO `filename`/`model`).
# The real M2/M3 modules write those SAME filenames to m2/ and m3/ with a richer
# schema that M3/M8 consume (cap["filename"], m2_output["model"]). If M11 wrote to
# m2/ or m3/ it would clobber a participant's genuine module output with an
# incompatible stub, and a later re-run of M8/M3 would crash with KeyError. Keeping
# M11's intermediates under m11/pipeline/ makes the demo self-contained and harmless.
m1_input_uri  = f"s3://{USER_BUCKET}/users/{PROFILE}/m1/"                     # real M1 (read-only)
step1_out_uri = f"s3://{USER_BUCKET}/users/{PROFILE}/m11/pipeline/step1_caption/"
step2_out_uri = f"s3://{USER_BUCKET}/users/{PROFILE}/m11/pipeline/step2_curate/"
step3_out_uri = f"s3://{USER_BUCKET}/users/{PROFILE}/m11/pipeline/step3_augment/"

# --- Instance / image for the steps --------------------------------------
# The three step scripts are pure Python (stdlib only) over small JSON — they do
# NO model inference, so they need no GPU. We run them on CPU (ml.m5.xlarge, which
# has processing-job quota; the g5* processing quota is 0 here) using the SDK's
# CPU sklearn container (Python 3, no GPU). This keeps the orchestration demo
# runnable and cheap. In production each step would use its own GPU image +
# instance (see docs/PIPELINE_M11.md); the DAG/dependencies/lineage are identical
# regardless of the per-step instance.
#
# NOTE: sklearn has NO "processing" image scope in the SDK image registry — the
# sklearn container is a single image reused across training/processing/inference
# and is resolved with NO image_scope argument (this is exactly how
# sagemaker.sklearn.processing.SKLearnProcessor resolves it internally). Passing
# image_scope="processing" raises `ValueError: Unsupported image scope: processing`
# and dead-stops this cell before any pipeline is defined. Omit it — the returned
# URI (…/sagemaker-scikit-learn:1.2-1-cpu-py3) is the correct CPU image.
STEP_INSTANCE_TYPE = "ml.m5.xlarge"
CPU_IMAGE_URI = sagemaker.image_uris.retrieve(
    framework="sklearn", region=region, version="1.2-1",
    instance_type=STEP_INSTANCE_TYPE,
)
print(f"Step instance: {STEP_INSTANCE_TYPE} (CPU) | image: {CPU_IMAGE_URI}")

# --- Step 1: Captioning (reads real M1, writes to m11-private namespace) ---
caption_processor = ScriptProcessor(
    image_uri=CPU_IMAGE_URI,
    role=role,
    instance_count=1,
    instance_type=STEP_INSTANCE_TYPE,
    command=["python3"],
    sagemaker_session=pipeline_session,
    tags=[{"Key": "Module", "Value": "M2-Caption"}]
)

step_caption = ProcessingStep(
    name="AV-Captioning",
    processor=caption_processor,
    code=str(SCRIPTS_DIR / "step1_caption.py"),
    inputs=[
        ProcessingInput(
            source=m1_input_uri,
            destination="/opt/ml/processing/input",
            s3_data_type="S3Prefix"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="captions",
            source="/opt/ml/processing/output",
            destination=step1_out_uri
        )
    ]
)

print("Step 1 (Captioning) defined:")
print(f"  Instance: {STEP_INSTANCE_TYPE}")
print(f"  Input:  {m1_input_uri}  (real M1, read-only)")
print(f"  Output: {step1_out_uri}")

# --- Step 2: Curation ---
curation_processor = ScriptProcessor(
    image_uri=CPU_IMAGE_URI,
    role=role,
    instance_count=1,
    instance_type=STEP_INSTANCE_TYPE,
    command=["python3"],
    sagemaker_session=pipeline_session,
    tags=[{"Key": "Module", "Value": "M3-Curate"}]
)

step_curate = ProcessingStep(
    name="Data-Curation",
    processor=curation_processor,
    code=str(SCRIPTS_DIR / "step2_curate.py"),
    inputs=[
        ProcessingInput(
            source=step1_out_uri,
            destination="/opt/ml/processing/input",
            s3_data_type="S3Prefix"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="curated",
            source="/opt/ml/processing/output",
            destination=step2_out_uri
        )
    ]
)
# Set dependency: curation depends on captioning
step_curate.add_depends_on([step_caption])

print("\nStep 2 (Curation) defined:")
print(f"  Instance: {STEP_INSTANCE_TYPE}")
print(f"  Input:  {step1_out_uri}")
print(f"  Output: {step2_out_uri}")
print(f"  Depends on: Step 1")

# --- Step 3: Augmentation ---
augment_processor = ScriptProcessor(
    image_uri=CPU_IMAGE_URI,
    role=role,
    instance_count=1,
    instance_type=STEP_INSTANCE_TYPE,
    command=["python3"],
    sagemaker_session=pipeline_session,
    tags=[{"Key": "Module", "Value": "M4-Augment"}]
)

step_augment = ProcessingStep(
    name="Data-Augmentation",
    processor=augment_processor,
    code=str(SCRIPTS_DIR / "step3_augment.py"),
    inputs=[
        ProcessingInput(
            source=step2_out_uri,
            destination="/opt/ml/processing/input",
            s3_data_type="S3Prefix"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="augmented",
            source="/opt/ml/processing/output",
            destination=step3_out_uri
        )
    ]
)
# Set dependency: augmentation depends on curation
step_augment.add_depends_on([step_curate])

print("\nStep 3 (Augmentation) defined:")
print(f"  Instance: {STEP_INSTANCE_TYPE}")
print(f"  Input:  {step2_out_uri}")
print(f"  Output: {step3_out_uri}")
print(f"  Depends on: Step 2")

In [ ]:
"""Create and register the SageMaker Pipeline"""

# Define pipeline
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        param_profile,
        param_input_bucket
    ],
    steps=[step_caption, step_curate, step_augment],
    sagemaker_session=pipeline_session
)

# Validate pipeline definition
try:
    pipeline_def = json.loads(pipeline.definition())
except Exception as e:
    raise RuntimeError(f"Failed to build the pipeline definition: {e}") from e
print(f"Pipeline Definition:")
print(f"  Name: {PIPELINE_NAME}")
print(f"  Steps: {len(pipeline_def['Steps'])}")
for step in pipeline_def['Steps']:
    print(f"    - {step['Name']} (type: {step['Type']})")
print(f"  Parameters: {len(pipeline_def['Parameters'])}")

# Create/update pipeline in SageMaker
print(f"\nCreating/updating pipeline: {PIPELINE_NAME}")
try:
    pipeline.upsert(role_arn=role)
except Exception as e:
    raise RuntimeError(
        f"Failed to register the pipeline (upsert): {e}\n"
        f"Check the execution role has sagemaker:CreatePipeline/UpdatePipeline "
        f"and iam:PassRole for this role."
    ) from e
print(f"Pipeline registered successfully.")
print(f"Console: https://{region}.console.aws.amazon.com/sagemaker/home#/pipelines/{PIPELINE_NAME}")

In [ ]:
"""Execute the pipeline"""

# Pre-flight: Step 1 mounts users/<profile>/m1/ as an S3Prefix. If M1 never
# ran, that prefix is empty and SageMaker fails the data-staging BEFORE the
# container starts — a slow, billable, confusing failure. Check first.
_m1 = s3.list_objects_v2(Bucket=USER_BUCKET, Prefix=f"users/{PROFILE}/m1/")
if _m1.get("KeyCount", 0) == 0:
    raise RuntimeError(
        f"No M1 output at s3://{USER_BUCKET}/users/{PROFILE}/m1/ — run M1 first. "
        f"M11 Step 1 captions the nuScenes scenes M1 selected "
        f"(selected_scenes.json); without them the pipeline has nothing to process."
    )

print(f"Starting pipeline execution: {PIPELINE_NAME}")
print(f"Parameters:")
print(f"  UserProfile: {PROFILE}")
print(f"  InputBucket: {USER_BUCKET}")
print(f"  (note: these pipeline parameters are recorded on the execution; the\n"
      f"   step S3 paths were fixed when the pipeline was defined this run.)")

start_time = time.time()

try:
    execution = pipeline.start(
        parameters={
            "UserProfile": PROFILE,
            "InputBucket": USER_BUCKET,
        }
    )
except Exception as e:
    raise RuntimeError(
        f"Failed to start the pipeline execution: {e}\n"
        f"The pipeline was upserted; check the execution role has "
        f"sagemaker:StartPipelineExecution and try again."
    ) from e

execution_arn = execution.arn
print(f"\nExecution started:")
print(f"  ARN: {execution_arn}")
print(f"\nWaiting for completion. Each CPU step spins up an ml.m5.xlarge, runs a")
print(f"few seconds of Python, and tears down — so the 3-step run is typically")
print(f"~5-10 min end-to-end (mostly per-step provisioning, not compute).")
print(f"  Monitor at: https://{region}.console.aws.amazon.com/sagemaker/home#/pipelines/{PIPELINE_NAME}/executions")

In [ ]:
"""Monitor execution status"""

def get_execution_status(execution):
    """Poll pipeline execution status and display step progress."""
    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]
    
    print(f"\nPipeline Status: {status}")
    print(f"Start time: {desc.get('CreationTime', 'N/A')}")
    
    # Get step details
    steps = execution.list_steps()
    print(f"\nStep Execution Status:")
    print("-" * 60)
    
    for step in steps:
        step_name = step["StepName"]
        step_status = step["StepStatus"]
        start = step.get("StartTime", "N/A")
        end = step.get("EndTime", "")
        
        status_icon = {
            "Succeeded": "DONE",
            "Failed": "FAIL",
            "Executing": ">>> ",
            "Starting": "... "
        }.get(step_status, "    ")
        
        duration = ""
        if start != "N/A" and end:
            dur_s = (end - start).total_seconds()
            duration = f" ({dur_s:.0f}s)"
        
        print(f"  [{status_icon}] {step_name:20s} | {step_status}{duration}")
    
    return status

# Poll until complete. Bounded + guarded: a transient API error or a hung
# Executing state must not spin forever or crash the poll (which would orphan
# the still-billing execution — you can Stop it from the console link above).
POLL_DEADLINE = time.time() + 30 * 60  # 30-min cap (steps normally ~5-10 min)
status = "Unknown"
while True:
    try:
        status = get_execution_status(execution)
    except Exception as e:
        print(f"  (status poll hiccup, will retry: {e})")
    if status in ["Succeeded", "Failed", "Stopped"]:
        break
    if time.time() > POLL_DEADLINE:
        print("\n  Poll timed out after 30 min — the execution may still be "
              "running. Check/Stop it at the console link above.")
        break
    print(f"\n  Waiting 30s before next check...")
    time.sleep(30)

total_pipeline_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Pipeline execution {status}!")
print(f"Total wall time: {total_pipeline_time:.0f}s ({total_pipeline_time/60:.1f} min)")

In [ ]:
"""Write pipeline execution metadata to S3"""

# Gather execution details
exec_desc = execution.describe()
steps_info = execution.list_steps()

pipeline_metadata = {
    "module": "M11_Pipeline_Automation",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "pipeline": {
        "name": PIPELINE_NAME,
        "execution_arn": execution_arn,
        "status": exec_desc["PipelineExecutionStatus"],
        "total_time_s": round(total_pipeline_time, 2),
        "creation_time": str(exec_desc.get("CreationTime", "")),
        "last_modified_time": str(exec_desc.get("LastModifiedTime", ""))
    },
    "steps": [
        {
            "name": step["StepName"],
            "status": step["StepStatus"],
            "start_time": str(step.get("StartTime", "")),
            "end_time": str(step.get("EndTime", ""))
        }
        for step in steps_info
    ],
    "pipeline_dag": {
        "step_1": {
            "name": "AV-Captioning",
            "represents_module": "M2 (captioning)",
            "instance": STEP_INSTANCE_TYPE,
            "input": "m1/ (selected_scenes.json — real M1, read-only)",
            "output": "m11/pipeline/step1_caption/captions.json"
        },
        "step_2": {
            "name": "Data-Curation",
            "represents_module": "M3 (curation)",
            "instance": STEP_INSTANCE_TYPE,
            "input": "m11/pipeline/step1_caption/",
            "output": "m11/pipeline/step2_curate/curated_captions.json",
            "depends_on": "step_1"
        },
        "step_3": {
            "name": "Data-Augmentation",
            "represents_module": "M4 (augmentation)",
            "instance": STEP_INSTANCE_TYPE,
            "input": "m11/pipeline/step2_curate/",
            "output": "m11/pipeline/step3_augment/augmented_captions.json",
            "depends_on": "step_2"
        }
    },
    "note": (
        "Demo runs all steps on CPU (ml.m5.xlarge) — the step scripts are pure "
        "Python over M1's scene metadata. Step outputs are written to an M11-private "
        "namespace (m11/pipeline/) so the demo never overwrites the real M2/M3/M4 "
        "module outputs (which use a different, richer schema consumed by M3/M8). "
        "Production steps would use GPU images (e.g. Cosmos Reason captioning); the "
        "DAG is identical. See docs/PIPELINE_M11.md."
    ),
}

# Upload metadata
output_key = f"{OUTPUT_PREFIX}pipeline_execution.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=output_key,
    Body=json.dumps(pipeline_metadata, indent=2),
    ContentType="application/json"
)
print(f"Pipeline metadata written to: s3://{USER_BUCKET}/{output_key}")

# Upload pipeline definition
definition_key = f"{OUTPUT_PREFIX}pipeline_definition.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=definition_key,
    Body=json.dumps(pipeline_def, indent=2),
    ContentType="application/json"
)
print(f"Pipeline definition written to: s3://{USER_BUCKET}/{definition_key}")

# Verify
print(f"\nOutput Validation:")
for key in [output_key, definition_key]:
    head = s3.head_object(Bucket=USER_BUCKET, Key=key)
    print(f"  OK: {key} ({head['ContentLength']} bytes)")

In [ ]:
"""Cost Analysis — pipeline orchestration + step execution"""

# Orchestration layer (this CPU notebook — negligible).
ORCH_INSTANCE = "ml.t3.medium"
ORCH_COST_PER_HOUR = 0.05  # USD
KRW_RATE = 1370

# What the demo ACTUALLY ran: 3 CPU processing steps on ml.m5.xlarge.
DEMO_STEP_INSTANCE = "ml.m5.xlarge"
DEMO_STEP_COST_HR = 0.23  # USD, ml.m5.xlarge on-demand (~same across regions)

# Pull real per-step durations from the execution we just monitored.
demo_steps = []
for st in execution.list_steps():
    start, end = st.get("StartTime"), st.get("EndTime")
    dur_s = (end - start).total_seconds() if (start and end) else 0.0
    demo_steps.append({"name": st["StepName"], "status": st["StepStatus"], "dur_s": dur_s})

demo_step_seconds = sum(s["dur_s"] for s in demo_steps)
demo_step_cost = DEMO_STEP_COST_HR * (demo_step_seconds / 3600)
orch_cost = ORCH_COST_PER_HOUR * (total_pipeline_time / 3600)
demo_total = demo_step_cost + orch_cost

print("=" * 60)
print("M11 Pipeline Automation — Cost Analysis")
print("=" * 60)
print()
print("--- This Workshop Demo (REAL, measured — CPU steps) ---")
print(f"Orchestration:   {ORCH_INSTANCE} (this notebook), ${orch_cost:.4f}")
print(f"Steps:           3x {DEMO_STEP_INSTANCE} @ ${DEMO_STEP_COST_HR:.2f}/hr")
for s in demo_steps:
    print(f"  - {s['name']:<18} {s['status']:<10} {s['dur_s']:.0f}s")
print(f"Step compute:    {demo_step_seconds:.0f}s total → ${demo_step_cost:.4f}")
print(f"Demo TOTAL:      ${demo_total:.3f} USD ({demo_total * KRW_RATE:.0f} KRW)")
print()
print("--- Production (CONCEPTUAL — GPU steps, not run here) ---")
# If the captioning/curation steps ran real VLMs they'd use GPU instances:
PROD = {
    "M2 Captioning": {"instance": "ml.g5.12xlarge", "cost_hr": 7.09, "est_min": 25},
    "M3 Curation":   {"instance": "ml.g5.12xlarge", "cost_hr": 7.09, "est_min": 15},
    "M4 Augmentation": {"instance": "ml.g5.xlarge", "cost_hr": 1.52, "est_min": 10},
}
prod_total = sum(v["cost_hr"] * (v["est_min"] / 60) for v in PROD.values())
for name, v in PROD.items():
    print(f"  {name:<16} {v['instance']:<16} ~{v['est_min']}min  ${v['cost_hr']*(v['est_min']/60):.2f}")
print(f"Production TOTAL (1 run): ~${prod_total:.2f} USD ({prod_total * KRW_RATE:,.0f} KRW)")
print()
print("--- Why the demo is CPU, and what a Pipeline gives you ---")
print("The step scripts are pure Python over M1's scene metadata, so they run on")
print("CPU for pennies. The VALUE M11 demonstrates is orchestration, not compute:")
print("  - reproducible DAG (same steps/order every run), step dependencies")
print("  - per-step instances start/stop automatically (no idle GPU cost)")
print("  - full lineage/audit trail; schedule on new data or cron")
print("Swapping the step image/instance to GPU (production) does not change the")
print("DAG — only the per-step compute. See docs/PIPELINE_M11.md.")
print("=" * 60)

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m11-orchestration")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")